# Start ×1 tegenover ×64 — tien leerbare schaalkeuzes, honderd epochs

**SVHN · CNN-8 · E4M3 → E2M3 → E2M1 · A32 · FP32-mastergewichten · B32**

Deze training onderzoekt de invloed van de **initialisatie** van de schaalparameters. De nieuwe run begint met alle 4.080 parameters op **θ = 0, dus factor ×1**. De eerdere volledige referentie begon op **θ = 6, dus ×64**. De tien factoren ×1, ×2, ×4, ×8, ×16, ×32, ×64, ×128, ×256 en ×512 blijven vrij leerbaar.

De nieuwe 100-epochrun start opnieuw vanaf de oorspronkelijke gewichten en BatchNorm-toestand. Het vijf-epochcheckpoint is geen warmstart. **Gewichten én schaalparameters leren tot epoch 100; er is geen bevriezing op 50.** De oorspronkelijke learningratemijlpalen50/75/90 blijven behouden.

×1 betekent **S_E2 = S_E4 per groep**, geen absolute schaal1. De E4M3-bronschaal wordt nog steeds uit de actuele gewichten bepaald; E2M3 en E2M1 delen hun lagere schaal. Dit is een initialisatiecontrole binnen de bestaande verhoudingsmethode, vóór de latere proef met onafhankelijke schalen.

**Bronnen:** eigen opgeslagen `history.json`, `run_report.json` en tabellen, de vastgelegde historische init×64-run en de afzonderlijke init×1-pilot. De notebook start geen training of nieuwe inferentie. Ontbrekende waarden worden niet ingevuld.

In [ ]:
from pathlib import Path
RUN_DIR = Path('C:\\Users\\renzo\\master-thesis-mx\\mx_runs\\mx_triple_learned_scale10_init1_reference100\\20261111_133600_075739_joint_check')
SAVE_FIGURES = False


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str): pass
    def display(value): print(value)

helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
if not helper_path.is_file():
    raise FileNotFoundError('Zet RUN_DIR op de resultatenmap van de init×1-training: ' + str(helper_path.parent))
spec = importlib.util.spec_from_file_location('_mx_scale10_init1_reference100_analysis', helper_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(pd.DataFrame(run.issues, columns=['Niveau', 'Melding']))


## 1. Volledig accuracyverloop

De drie precisiepanelen delen dezelfde y-as. Die volgt de werkelijk getoonde scores: nauwe verschillen worden ingezoomd; lage vroege accuracies blijven zichtbaar. Er wordt niets gladgestreken. Een historische curve stopt zodra daarvoor geen metingen beschikbaar zijn. De verticale stippellijnen markeren de oorspronkelijke LR-mijlpalen 50/75/90.

In [ ]:
display(Markdown(ns.comparison_markdown(run)))
figures['accuracy_verloop'] = ns.plot_initialization_accuracy(run)
display(figures['accuracy_verloop'])
plt.close(figures['accuracy_verloop'])


## 2. Eigen eindpunt en eigen beste checkpoint

Epoch100 is het vaste eindpunt en verschijnt pas als alle honderd epochs voor alle drie modi bevestigd zijn. Tijdens een deelrun staat de laatste afgeronde epoch apart. Het beste checkpoint is het **eerste hoogste E4M3-testresultaat** binnen de werkelijk gemeten nieuwe training. E2M3 en E2M1 worden op diezelfde epoch gerapporteerd; hun afzonderlijke maxima worden niet gecombineerd.

In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['eigen_checkpoints'] = ns.plot_checkpoint_accuracy(run)
display(figures['eigen_checkpoints'])
plt.close(figures['eigen_checkpoints'])


## 3. Vergelijking op epoch 100

Dit is de vooraf gekozen eindpuntvergelijking tussen beide initialisaties. Zij blijft leeg zolang niet beide volledige 100-epochresultaten bevestigd zijn. De punten gebruiken een ingezoomde as; er is geen afgekorte staaf die het verschil groter doet lijken.

In [ ]:
display(ns.initialization_checkpoint_table(run, 'endpoint').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_epoch100'] = ns.plot_initialization_checkpoints(run, 'endpoint')
display(figures['accuracy_epoch100'])
plt.close(figures['accuracy_epoch100'])


## 4. Vergelijking van de door E4M3 gekozen beste checkpoints

Beide varianten selecteren hun beste E4M3-checkpoint binnen **hetzelfde beschikbare epochvenster**. Bij een nieuwe deelrun van bijvoorbeeld50 epochs wordt de historische selectie eveneens tot50 beperkt. De gekozen epoch mag tussen varianten verschillen; alle drie precisies komen per variant steeds uit één checkpoint. Deze figuur gebruikt dezelfde asgrenzen als de epoch 100-vergelijking.

De officiële testset wordt volgens het bestaande protocol ook gebruikt voor checkpointselectie. De geselecteerde beste score is daarom geen onafhankelijke testschatting na selectie.

In [ ]:
display(ns.initialization_checkpoint_table(run, 'best').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_best_E4M3'] = ns.plot_initialization_checkpoints(run, 'best')
display(figures['accuracy_best_E4M3'])
plt.close(figures['accuracy_best_E4M3'])


## 5. Afzonderlijke vijf-epochpilot: vroege herhaling

De eerdere init×1-pilot is een eigen run. De nieuwe training begint opnieuw. Onderstaande tabel vergelijkt uitsluitend beide echte epoch 5-metingen, zodra de nieuwe run die epoch heeft bereikt. Het behouden CUDA-beleid garandeert geen bitexacte herhaling, ook bij dezelfde seed en datavolgorde. Pilotmetingen worden niet als epochs van de nieuwe run ingevoegd.

In [ ]:
display(ns.pilot_comparison_table(run).round(6))


## 6. Evolutie van alle tien factoren

Epoch0 is de opgeslagen beginstaat. Iedere kolom toont de gemeten verdeling van de 4.080 groepen aan die epochgrens. Alle tien factoren blijven zichtbaar, inclusief nulcategorieën. Bij lange reeksen staat het percentage in de kleurschaal; de gedetailleerde tellingen blijven beschikbaar in de tabellen.

In [ ]:
figures['factoren_per_epoch'] = ns.plot_factor_evolution(run)
display(figures['factoren_per_epoch'])
plt.close(figures['factoren_per_epoch'])


## 7. Aantallen en percentages bij de geselecteerde punten

De grafiek vergelijkt de beginstaat, de laatste nieuwe epoch en dezelfde historische epoch. De staven beginnen op nul. De tabellen bevatten daarnaast de verdeling bij het eigen E4M3-beste checkpoint. De gekozen factor is steeds een verhouding tot de actuele E4-bronschaal.

In [ ]:
display(ns.factor_distribution_table(run, 0).round({'Aandeel [%]': 4}))
display(ns.factor_distribution_table(run, run.observed_best_epoch).round({'Aandeel [%]': 4}))
display(ns.factor_distribution_table(run).round({'Aandeel [%]': 4}))
figures['factoren_aantallen'] = ns.plot_factor_counts(run)
display(figures['factoren_aantallen'])
plt.close(figures['factoren_aantallen'])


## 8. Continue θ-ontwikkeling

De optimizer wijzigt een continue FP32-parameter; de voorwaartse stap gebruikt de afgeronde integeroffset. Een parameter kan dus bewegen zonder al een andere factor te kiezen. Gemiddelde, P05–P95 en minimum/maximum laten deze beweging zien. Horizontale lijnen liggen op de afrondingsdrempels0,5;1,5;…;8,5.

In [ ]:
theta = ns.theta_summary_table(run)
display(theta.loc[theta.Laag.eq('Alle lagen')].round(6))
figures['theta_totaal'] = ns.plot_theta_evolution(run)
display(figures['theta_totaal'])
plt.close(figures['theta_totaal'])


## 9. Verschillen tussen lagen

De zes panelen delen hun y-as en tonen de continue parameters van de nieuwe init×1-training. Het bereik volgt de gemeten θ-waarden, zodat ook kleine verplaatsingen zichtbaar zijn. De tabel onderscheidt exact θ=0 van de afgeronde factor ×1.

In [ ]:
figures['theta_per_laag'] = ns.plot_theta_by_layer(run)
display(figures['theta_per_laag'])
plt.close(figures['theta_per_laag'])
display(theta.loc[theta.Laag.ne('Alle lagen'), ['Epoch', 'Laag', 'Gemiddelde θ', 'Min θ', 'Max θ', 'Exact θ=0 [%]', 'Factor ×1 [%]']].round(6))


In [ ]:
figures['theta_verdelingen'] = ns.plot_theta_distributions(run)
display(figures['theta_verdelingen'])
plt.close(figures['theta_verdelingen'])


## 10. Ondergrens, projecties en updates

Factor×1 betekent dat θ naar offset0 wordt afgerond; exact θ=0 betekent dat de parameter op de projectiegrens staat. Een update onder0 wordt op 0 geprojecteerd. Een groep kan binnen één epoch meerdere projecties of factorswitches ondergaan.

De gradientfiguur toont de grootste geregistreerde **laag-L2-norm** over batches, geen globale netwerkgradientnorm. Flips, unieke wisselende groepen en netto continue verplaatsing worden afzonderlijk bijgehouden.

In [ ]:
figures['grenzen_en_projecties'] = ns.plot_initialization_boundaries(run)
display(figures['grenzen_en_projecties'])
plt.close(figures['grenzen_en_projecties'])
figures['updates_en_gradients'] = ns.plot_initialization_updates(run)
display(figures['updates_en_gradients'])
plt.close(figures['updates_en_gradients'])


## 11. Losses en voorspelde klassen

E4M3 leert met harde label-CE. E2M3 leert van de detached E4M3-softmax; E2M1 leert van de detached E2M3-softmax. Deze trainobjectieven verschillen van harde test-CE en worden afzonderlijk getoond.

De voorspellingstellingen geven aan of een modus nog meerdere cijfers onderscheidt. Dit zijn geaggregeerde opgeslagen tellingen, geen nieuwe inferentie of gepaarde test op individuele beelden. Een lage accuracy of constante voorspelling is een onderzoeksuitkomst, geen automatische technische FAIL.

In [ ]:
figures['losses'] = ns.plot_objectives(run)
display(figures['losses'])
plt.close(figures['losses'])
display(ns.prediction_table(run, selected_only=True))
figures['voorspelde_klassen'] = ns.plot_prediction_diversity(run)
display(figures['voorspelde_klassen'])
plt.close(figures['voorspelde_klassen'])


## 12. Duur en opslagkosten

De gewijzigde beginfactor voegt geen opties, parameters of trainingspasses toe. Een flexibele bron met tien keuzes bewaart4 bits per groep:2.040 bytes bij4.080 groepen. De4.080 FP32-hulpparameters, gradients en twee Adam-momenten vragen analytisch 65.280 bytes, exclusief framework en tijdelijke buffers.

Dit is geen meting van hardware-energie of native MX-uitvoering. Bij een deelrun blijft een looptijdprojectie duidelijk onderscheiden van gemeten tijd; na100 epochs wordt de werkelijke som van epochtijden gerapporteerd.

In [ ]:
display(ns.initialization_cost_table(run))
display(ns.storage_cost_table(run))
figures['opslagkosten'] = ns.plot_storage_cost(run)
display(figures['opslagkosten'])
plt.close(figures['opslagkosten'])
display(Markdown(ns.timing_markdown(run)))
figures['epochtijden'] = ns.plot_timing(run)
display(figures['epochtijden'])
plt.close(figures['epochtijden'])


## 13. Technische controles en interpretatie

Onderstaande controles zijn tijdens de run opgeslagen. Deze notebook voert geen training, gradienttest of checkpointinferentie uit. De compacte exports bevatten één E4M3-bron met keuze-informatie; de lagere formaten worden uit die bron afgeleid.

**Te beoordelen:** verschuift de schaalverdeling vanuit×1 naar dezelfde regio als vanuit×64, en volgt de accuracy? Een verschil kan samenhangen met de start, geleidelijke optimalisatie, grensprojecties en gezamenlijke aanpassing van de gewichten. Eén run per initialisatie bewijst geen statistische superioriteit of globaal optimum. Ook100 epochs bewijzen op zichzelf geen convergentie.

Een afzonderlijke proef met onafhankelijke schalen per exponentformaat blijft een vervolgstap; die schalen worden hier niet ingevoerd. De oorspronkelijke onderzoeksdata blijven ongewijzigd.

In [ ]:
display(ns.audit_table(run))
display(ns.export_table(run))
display(Markdown(ns.scale_learning_markdown(run)))
display(pd.DataFrame(run.issues, columns=['Niveau', 'Melding']))


## Optioneel: figuren bewaren

De notebook blijft standaard alleen lezen en weergeven. Zet bovenaan `SAVE_FIGURES = True` om figuren en bronverwijzingen in een nieuwe analysemap te bewaren.

In [ ]:
if SAVE_FIGURES:
    destination = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', destination)
else:
    print('Alleen gelezen en weergegeven; bestaande resultaten zijn ongewijzigd.')
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
